# Movimiento Browniano Geométrico con Euler-Maruyama (diffeqpy)

Ecuación diferencial estocástica:

$$dX_t = \mu X_t\,dt + \sigma X_t\,dB_t$$

- **Parte determinista (deriva):** $\mu X_t\,dt$
- **Parte con ruido (difusión):** $\sigma X_t\,dB_t$, con $B_t$ movimiento browniano estándar

Método de Euler-Maruyama:

$$X_{n+1} = X_n + \mu X_n\,\Delta t + \sigma X_n\sqrt{\Delta t}\,Z_n, \qquad Z_n \sim N(0,1)$$

**Cómo usar este cuaderno:** cambia los valores de la celda de *Parámetros* y ejecuta todo con
**Run → Run All Cells**, o celda por celda con **Shift + Enter**.

## 1. Preparación

In [ ]:
import os
# Asegura que Python encuentre Julia (instalada con juliaup), aunque Jupyter se abra sin ese PATH
os.environ["PATH"] = os.path.expanduser("~/.juliaup/bin") + os.pathsep + os.environ["PATH"]

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from diffeqpy import de   # la primera vez tarda ~1 minuto (Julia carga los paquetes)
print("diffeqpy listo")

## 2. Parámetros  ✏️ *(modifica aquí)*

Valores de la clase: $\mu = 0.25$, $\sigma = 1$, $X_0 = 1$, $T = 10$, 20 trayectorias.

Ideas para experimentar:
- `X0 = 0` → todas las trayectorias quedan en 0 (el ruido es multiplicativo). Usa `escala_log = False`.
- `sigma = 0.5` → $\mu - \sigma^2/2 > 0$: la mediana crece.
- `sigma = 0.7071` → $\mu = \sigma^2/2$: la mediana queda plana.
- `dt = 0.1` → paso grande, Euler-Maruyama pierde precisión.
- `M = 2000` → la media muestral se acerca (lentamente) a la teórica.

In [ ]:
mu    = 0.25     # deriva
sigma = 1.0      # volatilidad
X0    = 1.0      # condición inicial
T     = 10.0     # tiempo final
dt    = 0.01     # paso de tiempo
M     = 20       # número de trayectorias
semilla = 1      # cambia la semilla para obtener otras trayectorias
escala_log = True

## 3. Definición de la EDE

En diffeqpy se define $dX = f(X,p,t)\,dt + g(X,p,t)\,dB$:

In [ ]:
def f(u, p, t):          # deriva (parte determinista): mu * X
    return p[0] * u

def g(u, p, t):          # difusión (parte con ruido): sigma * X
    return p[1] * u

prob = de.SDEProblem(f, g, X0, (0.0, T), (mu, sigma))

## 4. Simulación con Euler-Maruyama

In [ ]:
X = np.array([np.array(de.solve(prob, de.EM(), dt=dt, saveat=dt, seed=semilla + k).u)
              for k in range(M)])               # forma (M, número de pasos)
t = np.linspace(0.0, T, X.shape[1])

media_teo   = X0 * np.exp(mu * t)                       # E[X_t]
mediana_teo = X0 * np.exp((mu - sigma**2 / 2) * t)      # Med[X_t]

print(f"mu - sigma^2/2 = {mu - sigma**2/2:.4f}  ->  "
      f"{'la mediana decrece' if mu - sigma**2/2 < 0 else 'la mediana crece o es constante'}")
print(f"En t = T:  media teórica   = {media_teo[-1]:.4f}   mediana teórica   = {mediana_teo[-1]:.4f}")
print(f"           media muestral  = {X[:, -1].mean():.4f}   mediana muestral  = {np.median(X[:, -1]):.4f}")

## 5. Gráfica de las trayectorias (estilo de la clase)

In [ ]:
plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm", "font.size": 13})
fondo = "#fbf7ef"
fig, ax = plt.subplots(figsize=(8, 5), facecolor=fondo)

for xi in X:
    ax.plot(t, xi, color="0.65", lw=0.8)
ax.plot(t, media_teo,   color="#a3262a", lw=2.5, label=r"media $e^{\mu t}$")
ax.plot(t, mediana_teo, color="#2a63c6", lw=2.5, label=r"mediana $e^{(\mu-\sigma^2/2)t}$")

if escala_log:
    ax.set_yscale("log")
ax.set_xlim(0, T)
ax.set_xlabel(r"$t$")
ax.set_title(f"{M} trayectorias (Euler-Maruyama)", color="#1f3b6e", fontweight="bold",
             fontfamily="sans-serif", fontsize=16, pad=12)
ax.grid(True, color="0.9")
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
ax.legend(loc="lower left", frameon=False)
plt.tight_layout()
plt.savefig("resultado_mbg_notebook.png", dpi=150, facecolor=fondo)
plt.show()

## 6. Comparación con la solución determinista

Sin ruido ($\sigma = 0$) la ecuación es $\dfrac{dx}{dt} = \mu x$, con solución $x(t) = X_0 e^{\mu t}$,
que coincide con la media teórica $E[X_t]$. Aquí se resuelve numéricamente con diffeqpy y se compara
con el promedio de las trayectorias simuladas.

In [ ]:
prob_ode = de.ODEProblem(f, X0, (0.0, T), (mu, sigma))
sol_ode  = de.solve(prob_ode, de.Tsit5(), saveat=dt)
x_det    = np.array(sol_ode.u)

promedio = X.mean(axis=0)
print(f"Máx |EDO numérica - X0·e^(mu t)| = {np.max(np.abs(x_det - media_teo)):.3e}")
print(f"En t = T:  determinista = {x_det[-1]:.4f}   promedio de {M} trayectorias = {promedio[-1]:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
for xi in X:
    ax.plot(t, xi, color="0.75", lw=0.7)
ax.plot(t, promedio, "r--", lw=2, label=f"Promedio de las {M} trayectorias")
ax.plot(t, x_det, "k", lw=2.5, label=r"Determinista $x(t)=X_0e^{\mu t}$")
if escala_log:
    ax.set_yscale("log")
ax.set_xlabel(r"$t$"); ax.set_ylabel(r"$X_t$")
ax.set_title("Euler-Maruyama vs. solución determinista")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()
plt.show()